# Data Re-Uploading QNN for Coupled Texture Defects

This notebook implements the most practical standard QNN option: **data re-uploading**.

Instead of loading each feature once, the circuit loads the same features repeatedly across trainable layers. This gives a small circuit a richer nonlinear decision boundary. For a hackathon, it is easier to explain and implement than a large generic QNN, while still being recognizably neural-network-like.

## 1. Imports

The key Qiskit object is `SamplerQNN`, wrapped by `NeuralNetworkClassifier`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import zz_feature_map
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms.optimizers import COBYLA, SPSA
from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.neural_networks import SamplerQNN

SEED = 8398
np.random.seed(SEED)

## 2. Helpers and Dataset

The dataset is the same industrial micro-defect proxy used in the QCNN notebook, so the approaches can be compared directly.

In [ ]:
def generate_microdefect_latents(n_samples=80, seed=SEED + 100):
    """Synthetic proxy for calibrated industrial texture features.

    Think of each row as four compact features produced by an inspection front-end:
    two local texture phase measurements and two orientation/defect-coupling scores.
    The label depends on coupled factors, not a single obvious threshold.
    """
    rng = np.random.default_rng(seed)
    X = rng.uniform(0.0, 2 * np.pi, size=(n_samples, 4))
    score = np.sin(X[:, 0]) * np.cos(X[:, 1]) + np.sin(X[:, 2] + X[:, 3])
    y = (score > np.median(score)).astype(int)
    return X, y, score


def latents_to_microtexture_images(X, size=8, seed=SEED):
    """Render latent texture factors as small grayscale inspection patches."""
    rng = np.random.default_rng(seed)
    grid = np.linspace(0, 2 * np.pi, size)
    rr, cc = np.meshgrid(grid, grid, indexing="ij")
    images = []

    for x in X:
        img = (
            0.50
            + 0.16 * np.sin(rr + x[0])
            + 0.13 * np.cos(cc + x[1])
            + 0.10 * np.sin(rr + cc + x[2])
            + 0.08 * np.cos(rr - cc + x[3])
        )
        img += rng.normal(0.0, 0.025, size=(size, size))
        images.append(np.clip(img, 0.0, 1.0))

    return np.array(images)


def show_microtextures(images, labels, scores=None, n_show=12, cols=6):
    order = np.arange(min(n_show, len(images)))
    rows = int(np.ceil(len(order) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.7))
    axes = np.ravel(axes)

    for ax, idx in zip(axes, order):
        ax.imshow(images[idx], cmap="gray_r", vmin=0, vmax=1)
        label = "DEFECT" if labels[idx] else "clean"
        suffix = "" if scores is None else f"\nscore={scores[idx]:.2f}"
        ax.set_title(f"#{idx} {label}{suffix}", fontsize=8, color="crimson" if labels[idx] else "black")
        ax.set_xticks([])
        ax.set_yticks([])

    for ax in axes[len(order):]:
        ax.axis("off")

    fig.tight_layout()
    plt.show()


def evaluate_predictions(name, y_true, y_pred, *, show_confusion=True):
    print(f"\n{name}")
    print("balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
    print("defect F1 score  :", f1_score(y_true, y_pred, zero_division=0))
    print(classification_report(y_true, y_pred, target_names=["clean", "defect"], zero_division=0))

    if show_confusion:
        ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=["clean", "defect"])
        plt.title(name)
        plt.show()


def evaluate_classifier(name, model, X_eval, y_eval, *, show_confusion=True):
    y_pred = model.predict(X_eval)
    evaluate_predictions(name, y_eval, y_pred, show_confusion=show_confusion)
    return y_pred


def balanced_subset(X_data, y_data, n_per_class=18, seed=SEED):
    rng = np.random.default_rng(seed)
    chosen = []
    for cls in np.unique(y_data):
        cls_idx = np.flatnonzero(y_data == cls)
        take = min(n_per_class, len(cls_idx))
        chosen.extend(rng.choice(cls_idx, size=take, replace=False).tolist())
    rng.shuffle(chosen)
    return X_data[chosen], y_data[chosen]


def readout_q0(measured_integer):
    """Use qubit 0 as the binary classifier readout."""
    return measured_integer & 1


def parity_readout(measured_integer):
    """Map a measured computational-basis state to class 0 or 1 by parity."""
    return bin(measured_integer).count("1") % 2



def build_data_reuploading_circuit(n_qubits=4, layers=1):
    """Build a small data re-uploading QNN.

    Each layer loads the same data again, applies trainable rotations, and entangles
    neighboring qubits. Re-uploading lets a small number of qubits build a richer
    nonlinear decision boundary.
    """
    x = ParameterVector("x", n_qubits)
    theta = ParameterVector("theta", layers * n_qubits * 2)
    qc = QuantumCircuit(n_qubits)

    t = 0
    for layer in range(layers):
        for q in range(n_qubits):
            qc.ry(x[q], q)
            qc.rz(x[(q + 1) % n_qubits], q)

        for q in range(n_qubits):
            qc.ry(theta[t], q)
            t += 1
            qc.rz(theta[t], q)
            t += 1

        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)
        qc.cx(n_qubits - 1, 0)

    return qc, list(x), list(theta)

## 3. Generate the Data

The label depends on coupled texture factors. A single-feature threshold should not be enough.

In [ ]:
N_SAMPLES = 80
X, y, scores = generate_microdefect_latents(n_samples=N_SAMPLES, seed=SEED + 100)
images = latents_to_microtexture_images(X, size=8, seed=SEED)

print("feature shape:", X.shape)
print("class balance:", {"clean": int((y == 0).sum()), "defect": int((y == 1).sum())})
show_microtextures(images, y, scores=scores, n_show=12, cols=6)

## 4. Split and Baselines

A data re-uploading QNN should be compared against simple classical baselines. If classical baselines win decisively, this should become a methods comparison rather than a quantum-advantage claim.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y,
)

baseline_models = {
    "Linear SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="linear", class_weight="balanced", random_state=SEED)),
    ]),
    "RBF SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="rbf", class_weight="balanced", random_state=SEED)),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=SEED,
    ),
}

for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    evaluate_classifier(name, model, X_test, y_test, show_confusion=False)

## 5. Build a Data Re-Uploading Circuit

`REUPLOAD_LAYERS=1` is deliberately modest. More layers can help, but can also make optimization worse.

In [ ]:
REUPLOAD_LAYERS = 1

reupload_circuit, input_params, weight_params = build_data_reuploading_circuit(
    n_qubits=4,
    layers=REUPLOAD_LAYERS,
)

print("qubits:", reupload_circuit.num_qubits)
print("trainable weights:", len(weight_params))
print("circuit depth:", reupload_circuit.depth())
reupload_circuit.draw(output="mpl")

## 6. Train the Data Re-Uploading QNN

This uses parity readout because it worked well for this compact circuit in testing. Try `readout_q0` too; readout choice is a real QNN design knob.

In [ ]:
REUPLOAD_TRAIN_PER_CLASS = 18
REUPLOAD_MAXITER = 80

X_qnn_train, y_qnn_train = balanced_subset(
    X_train,
    y_train,
    n_per_class=REUPLOAD_TRAIN_PER_CLASS,
    seed=SEED,
)

sampler_qnn = SamplerQNN(
    circuit=reupload_circuit,
    sampler=StatevectorSampler(seed=SEED),
    input_params=input_params,
    weight_params=weight_params,
    interpret=parity_readout,
    output_shape=2,
)

rng = np.random.default_rng(SEED)
initial_point = rng.uniform(-0.2, 0.2, size=len(weight_params))

reupload_classifier = NeuralNetworkClassifier(
    neural_network=sampler_qnn,
    optimizer=COBYLA(maxiter=REUPLOAD_MAXITER),
    initial_point=initial_point,
)

reupload_classifier.fit(X_qnn_train, y_qnn_train)
evaluate_classifier("Data re-uploading SamplerQNN", reupload_classifier, X_test, y_test)

## 7. Optional SPSA Run

SPSA is useful to discuss because it is noise-tolerant, but it can be slower and more variable. Leave it off until the COBYLA path is stable.

In [ ]:
RUN_SPSA = False

if RUN_SPSA:
    rng = np.random.default_rng(SEED + 1)
    spsa_classifier = NeuralNetworkClassifier(
        neural_network=sampler_qnn,
        optimizer=SPSA(maxiter=REUPLOAD_MAXITER),
        initial_point=rng.uniform(-0.2, 0.2, size=len(weight_params)),
    )
    spsa_classifier.fit(X_qnn_train, y_qnn_train)
    evaluate_classifier("Data re-uploading SamplerQNN with SPSA", spsa_classifier, X_test, y_test)
else:
    print("Set RUN_SPSA = True to try SPSA.")

## 8. How to Interpret This Direction

Data re-uploading is the best fallback if a QCNN-style circuit feels too custom. It is compact, explainable, and grounded in known QNN theory. The weakness is trainability: several seeds are needed before trusting the result.

## References and AI Tools Disclosure

This notebook was drafted with OpenAI Codex/GPT-5 assistance in the local hackathon workspace. The team should treat the code and results as AI-assisted prototypes and verify claims before presenting them.

AI-assisted notebooks in this `main_challenge` folder:

- `01_defect_classical_hardness_audit.ipynb`
- `02_quantum_kernel_teacher_defects.ipynb`
- `03_projected_quantum_kernel_features.ipynb`
- `04_qnn_vs_kernel_bakeoff.ipynb`
- `05_qcnn_industrial_microdefects.ipynb`
- `06_data_reuploading_qnn_microdefects.ipynb`
- `07_qnn_kernel_pivot_scoreboard.ipynb`
- `08_imbalanced_rare_defect_qnn.ipynb`
- `09_normal_only_anomaly_detection.ipynb`

References used for the quantum-ML direction:

- Cong, Choi, and Lukin, "Quantum convolutional neural networks," Nature Physics 15, 1273-1278 (2019): https://www.nature.com/articles/s41567-019-0648-8
- Pérez-Salinas et al., "Data re-uploading for a universal quantum classifier," Quantum 4, 226 (2020): https://doi.org/10.22331/q-2020-02-06-226 and https://arxiv.org/abs/1907.02085
- McClean et al., "Barren plateaus in quantum neural network training landscapes," Nature Communications 9, 4812 (2018): https://www.nature.com/articles/s41467-018-07090-4
- Havlíček et al., "Supervised learning with quantum-enhanced feature spaces," Nature 567, 209-212 (2019): https://www.nature.com/articles/s41586-019-0980-2
- Schölkopf et al., "Estimating the Support of a High-Dimensional Distribution," Neural Computation 13(7), 1443-1471 (2001): https://doi.org/10.1162/089976601750264965
- Ngairangbam, Spannowsky, and Takeuchi, "Anomaly detection in high-energy physics using a quantum autoencoder," Physical Review D 105, 095004 (2022): https://arxiv.org/abs/2112.04958
- Quantum Patch-Based Autoencoder for Anomaly Segmentation: https://arxiv.org/abs/2404.17613
- Quantum Machine Learning Algorithms for Anomaly Detection: a Review: https://arxiv.org/abs/2408.11047
- Qiskit Machine Learning `SamplerQNN` documentation: https://qiskit-community.github.io/qiskit-machine-learning/stubs/qiskit_machine_learning.neural_networks.SamplerQNN.html